# XGBoost — Student Test Score Prediction
MSE 546 Final Project — Group 3

XGBoost regressor to predict student exam scores.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [ ]:
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')
sample_sub = pd.read_csv('../data/sample_submission.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
train_df.head()

## 3. Quick EDA

In [ ]:
print(train_df.info())
print('\n--- Missing Values ---')
print(train_df.isnull().sum())
print('\n--- Target Stats ---')
print(train_df['exam_score'].describe())

## 4. Preprocessing

Encode categorical features using Label Encoding. We fit encoders on the combined train+test data to avoid unseen labels at prediction time.

In [ ]:
# Identify categorical columns
categorical_cols = ['gender', 'course', 'internet_access', 'sleep_quality',
                    'study_method', 'facility_rating', 'exam_difficulty']

# Label encode each categorical column (fit on combined data)
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))
    label_encoders[col] = le

print('Encoding complete.')
train_df.head()

## 5. Prepare Features and Target

In [ ]:
# Features (drop id and target)
feature_cols = [c for c in train_df.columns if c not in ['id', 'exam_score']]

X = train_df[feature_cols]
y = train_df['exam_score']
X_test = test_df[feature_cols]

print(f'Features: {feature_cols}')
print(f'X shape: {X.shape}, y shape: {y.shape}, X_test shape: {X_test.shape}')

## 6. Train/Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}, Validation: {X_val.shape}')

## 7. Train XGBoost Model

In [ ]:
model = XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    early_stopping_rounds=50,
    verbosity=1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

print(f'\nBest iteration: {model.best_iteration}')

## 8. Evaluate on Validation Set

In [ ]:
y_pred_val = model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
r2 = r2_score(y_val, y_pred_val)

print('=== Validation Results ===')
print(f'MAE:  {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R²:   {r2:.4f}')

## 9. Feature Importance

In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance.to_string(index=False))

## 10. Generate Submission

In [ ]:
# Predict on test set
test_predictions = model.predict(X_test)

# Create submission DataFrame
submission = pd.DataFrame({
    'id': test_df['id'],
    'exam_score': test_predictions
})

# Save to submission folder
import os
os.makedirs('../submission', exist_ok=True)
submission.to_csv('../submission/xgboost_submission.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(f'Saved to ../submission/xgboost_submission.csv')
submission.head()